<a href="https://colab.research.google.com/github/lamrosset/Atomistic-ML-tutorial/blob/main/oxmlip_tutorial_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Molecular Dynamics (MD) with MLIPs

## An introduction to MD

There are many existing codes for molecular dynamics simulations. In this notebook we will use the [ASE](https://wiki.fysik.dtu.dk/~askhl/ase-doc/tutorials/md/md.html?highlight=molecular%20dynamics) python library, which provides an interface to a great number of MLIP architectures. The [LAMMPS](https://www.lammps.org/) code is the other most popular code for MD simulations, which can handle much more complex simulation protocols.

Molecular dynamics simulations model the time evolution of a system of atoms by integrating their equations of motion, following classical mechanics and Newton's second law of motion:

$$ \mathbf{F}_i=m_i\cdot \mathbf{a}_i $$

where forces are derived as the gradients of the potential energy surface ($V$) with respect to atomic displacements:

$$\mathbf{F}_i=-\nabla_{\mathbf{r}_i}V(\mathbf{r}_1,\ldots,\mathbf{r}_N)$$


The velocity Verlet algorithm is the most popular implementation of time integration. It allows for the computation of positions, velocities and accelerations on the atoms at a time $t+\Delta t$ from the corresponding values at a time $t$.

Further details of the method are available at :
- Frenkel, Daan, and Berend Smit. *Understanding molecular simulation: from algorithms to applications*. Academic Press, 2002.
- Ercolessi, *A Molecular Dynamics Primer*. Spring College in Computational Physics, ICTP, Trieste, June 1997.

The ``graph-pes`` package, used in the previous tutorial to train and fine tune various potentials, includes a [plug-in](https://vldgroup.github.io/graph-pes/tools/ase.html#) to the ASE package, and to ASE's MD implementation.


As before, we install ``graph-pes`` using pip and `mace-torch` to access the foundation models:


In [ ]:
!pip install graph-pes mace-torch

## A note on starting structures

The chosen starting structural model for a MD simulation is usually highly dependent on the application of interest.
This may be a crystalline structure, a liquid, a disordered structure, or simply a randomly packed cell.

A number of structures are available to :

In [101]:
%%bash
mkdir -p structures

In [ ]:
%%bash

wget https://github.com/lamrosset/Atomistic-ML-tutorial/blob/main/structures/amorph-C_200_1.2.xyz
wget https://github.com/lamrosset/Atomistic-ML-tutorial/blob/main/structures/amorph-C_200_2.4.xyz
wget https://github.com/lamrosset/Atomistic-ML-tutorial/blob/main/structures/amorph-C_200_3.5.xyz
wget https://github.com/lamrosset/Atomistic-ML-tutorial/blob/main/structures/random-C_200_1.2.xyz
wget https://github.com/lamrosset/Atomistic-ML-tutorial/blob/main/structures/random-C_200_2.4.xyz
wget https://github.com/lamrosset/Atomistic-ML-tutorial/blob/main/structures/random-C_200_3.5.xyz
wget https://github.com/lamrosset/Atomistic-ML-tutorial/blob/main/structures/C_diamond.poscar

mv *.xyz ./structures
mv *.poscar ./structures

The structures should now be available in the ``./structures`` folder, covering a range of phases and densities.

## Loading the potentials

We will use all three models you have handled in the previous tutorial, that is:
- the MACE-MP0 model, here loaded into the `mp0` object
- the fine-tuned MACE-MP0 model, here loaded into the `mp0_ft` object
- the bespoke MACE model, here loaded into the `bespoke` object

In [ ]:
import torch
from graph_pes.interfaces import mace_mp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the pre-trained MACE-MP0 model
mp0 = mace_mp("small").eval().to(device)


In [ ]:
from graph_pes.models import load_model
import torch

# Load your fine-tuned model
mp0_ft = (
    load_model("path-to-fine-tune/model.pt")  # load the model, edit to the correct path
    .to(device)  # move to GPU if available
    .eval()  # set to evaluation mode
)

# Load your bespoke model
bespoke = (
    load_model("bespoke-train.pt")  # load the model, edit to the correct path
    .to(device)  # move to GPU if available
    .eval()  # set to evaluation mode
)

## Annealing simulations



We will first consider the simplest possible example of molecular dynamics simulations: an annealing simulation, where the system is held at a chosen temperature for a given time.


In [ ]:
from ase.io import read

# Choose a starting structure
path_to_structure='./structures/C_diamond.poscar' # specify the path to the structure

initial_structure=read(path_to_structure) # load the structure as an ASE Atoms object


In [ ]:
import ase
from ase.md.velocitydistribution import (MaxwellBoltzmannDistribution, ZeroRotation, Stationary)
from ase.md import MDLogger
from ase.md.nvtberendsen import NVTBerendsen
from ase.io import read, write
import os

# Specify parameters of the MD simulation
Tinit = 300  # initial temperature in Kelvin

md_params = {
    "timestep": 1 * ase.units.fs,  # MD timestep in femtoseconds, 1 fs is typical for MLIPs
    "taut": 100 * 0.5 * ase.units.fs,  # thermostat time constant
}
total_md_steps = 100  # make sure change this to match the appropriate time scale of your system.

# Attach a calculator (ie, a potential to drive the simulation) to the initial structure
initial_structure.calc=mp0.ase_calculator()

# Initialise velocities
MaxwellBoltzmannDistribution(initial_structure, temperature_K=Tinit)
Stationary(initial_structure)
ZeroRotation(initial_structure)

# Initialise dynamics object
dyn = NVTBerendsen(initial_structure, temperature_K=Tinit, **md_params)

# Make a directory where the outputs will be saved
os.makedirs('annealing', exist_ok=True)

# Write trajectory function
def write_frame():
    dyn.atoms.write(
        f"./annealing/trajectory.xyz", append=True
    )  # make sure the extension is xyz. Make sure to save the path of each trajectory file to avoid overwriting.


dyn.attach(write_frame, interval=100)  # set the frequency of writing to trajectory file

# Set up the logger
dyn.attach(
    MDLogger(
        dyn,  # dynamics object
        initial_structure,  # intial configuration
        f"./annealing/log.log", # path to where the log file is written
        peratom=True,
        mode="a",
    ),
    interval=10,  # frequency of writing the log
)

# Run the MD
dyn.run(total_md_steps)

True

###  Visualising MD trajectories

If you don't already have it installed, `OVITO` is a fantastic tool for visualisations. You can install it for free by following the instructions on the [OVITO website](https://www.ovito.org/#download).

If you cannot download `OVITO`, the trajectories can also be visualized as follows:



In [ ]:
from ase.visualize import view

# Load the full MD trajectory
traj = ase.io.read('./annealing/trajectory.xyz', index=':')

# Visualizing a single snapshot
view(traj[0], viewer='x3d') # first frame

In [ ]:
view(traj[-1], viewer='x3d') # last frame

In [ ]:
# alternatively, you can make a short movie of the full trajectory
ase.io.write('./annealing/movie.gif', traj)

# The movie can be viewed by opening the .gif file in a separate window

## Geometry optimisation

Geometry optimisations describe the task of finding the lattice vectors and atom positions that minimize the energy of a system.

A variety of optimisers are implemented in ASE as described [here](https://wiki.fysik.dtu.dk/~askhl/ase-doc/ase/optimize.html#module-ase.optimize), here we use the local LBFGS optimiser.

In [ ]:
from ase.optimize import LBFGS

initial_structure=read('./structures/C_diamond.poscar')
initial_structure.calc=mp0.ase_calculator()

opt = LBFGS(initial_structure)
opt.run(fmax=0.05, steps=100)  # run the optimiser until forces are below 0.05 eV/Ang, or 100 steps are reached.

# Make a directory where the outputs will be saved
os.makedirs('geometry_opt', exist_ok=True)

# write the optimised structure to a file
write("geometry_opt/opt_diamond.xyz", initial_structure)


### An example - vacancy in diamond





Geometry optimisations are often used to compute the energy and geometry of defects in crystalline systems, as done with quantum mechanical methods.
You will now apply the carbon potentials to computing the energy of a vacancy defect in diamond carbon.


Here is some additional help:
 - Guidance on how to delete an atom from a structure is available in the ASE documentation [here](https://wiki.fysik.dtu.dk/~askhl/ase-doc/ase/atoms.html).

- As discussed in Corsetti and Mostofi, *Phys. Rev. B*, **84**, 035209 (2011), the energy of the defect, $E_f$, is obtained from the energy of the bulk structure $E_{bulk}$ with $N$ atoms and the energy of the cell with the vacancy $E_{vac}$:

$$ E_{f}=E_{vac} - \frac{(N-1)}N E_{bulk} $$

In [ ]:
# Load the initial diamond structure, and attach a calculator to the structure to run the dynamics

ini_energy = #energy of the bulk structure
N = # number of atoms in the bulk structure

# Create a vacancy by deleting an atom in the relaxed initial structure


# Now run the relaxation


# Make sure to save the structure for analysis

# Compute the energy of the defect using the formula given above :
end_energy = #energy of the relaxed vacancy structure
defect_en= #
print(f"Defect energy: {defect_en} eV")


In [ ]:
traj=read('geometry_opt/vacancy_traj.traj', index=':')
ase.io.write('./geometry_opt/vacancy.gif', traj)

The estimated vacancy formation energy from literature ranges from **6.8 to 7 eV**. How do the different models perform? Can you rationalise these results?


An additional consideration around vacancies is the relaxed geometry. Vacancies in diamond cause inward relaxation behavior from the neighboring atoms, which may be incorrectly modelled as outward relaxation without sufficient training data.

You may analyse this with python code, or simply in OVITO.

## Graphitisation

We now turn to the testing the models on disordered configurations.
One relevant task for amorphous carbon is modelling graphetisation.

A general protocol for graphetisation is:
- start from an amorphous carbon configuration
- ramp the temperature from 300 K to a high temperature between 2000-4000 K
- anneal for hundreds of ps at that target temperature



In [ ]:
import numpy as np
from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary, ZeroRotation
from ase.md.nvtberendsen import NVTBerendsen
from ase.md import MDLogger
from ase.io import read, write
from ase.io.trajectory import Trajectory

# Parameters
Tinit = 300      # Initial temperature (K)
Tmelt = 6000     # Final temperature (K)

timestep = 1.0 * units.fs  # Time step
taut = 50.0 * units.fs     # Thermostat time constant
interval = 100             # Write output every 100 steps

# Simulation steps (adjust as needed)
heating_steps = 3000
anneal_hold_steps = 5000

# Load initial structure and attach calculator
initial_frame=read('structures/C_diamond.poscar')
initial_frame.calc=bespoke.ase_calculator()

MaxwellBoltzmannDistribution(initial_frame, temperature_K=Tinit)
Stationary(initial_frame)
ZeroRotation(initial_frame)

# Make a directory where the outputs will be saved
directory=f'graphitisation'
os.makedirs(directory, exist_ok=True)


# Initialize trajectory and logger
traj = f"{directory}/trajectory.xyz"
logfile = f"{directory}/md.log"

def write_frame():
    dyn.atoms.write(
        f"./{directory}/trajectory.xyz", append=True
    )  # make sure the extension is xyz. Make sure to save the path of each trajectory file to avoid overwriting.

# === Create dynamics object ===
dyn = NVTBerendsen(initial_frame, temperature_K=Tinit, timestep=timestep, taut=taut)
dyn.attach(write_frame, interval=interval)  # set the frequency of writing to trajectory file
dyn.attach(MDLogger(dyn, initial_frame, logfile, mode="a", peratom=True), interval=interval)

# === Heating phase ===
print(f"Heating from {Tinit} K to {Tmelt} K")
heating_temps = np.linspace(Tinit, Tmelt, heating_steps // 10)
for T in heating_temps:
  #print(T)
  dyn.set_temperature(T)
  dyn.run(10)

# === Annealing / hold phase ===
print(f"Annealing at {Tmelt} K...")
dyn.set_temperature(Tmelt)
dyn.run(anneal_hold_steps)

print("Graphitisation complete.")


In [ ]:
view(dyn.atoms, viewer='x3d')

Now look at your structure - is it graphetised ?

From tracking a variable of your choice over the progress of the simulation, can you estimate a window for when it graphetised?

## Melt-quench simulations

The most general way of preparing bulk amorphous structures is the 'melt-quench' protocol, where a random structure is heated above its melting point (> 4000 K), then cooled down to room temperature at a chosen quench rate, typically of the order of $10^{15}$ to $10^{12}$ K/s.

In [ ]:
# Adapt the annealing and graphitisation codes to simulate a melt quench simulation



Now some time to explore further:

- How does the structure change throughout the simulation? You could track the number of sp2 and sp3 carbon environments, the average bond length etc.

- How is the structure affected by the quench rate ?

- How is the structure affected by the density of the random box?